In [13]:
import numpy as np
from scipy import constants
import pandas as pd
pd.set_option('display.width', 10000) # Adjust for desired width
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) # display full content in a column
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker
import itertools
import math

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)


3540698434791077.0
[1.47e+08 1.40e+08]


In [14]:
import time

N_max = 15
N_values = np.arange(2, N_max)
w0 = 1e-6
P_list = np.linspace(1e-6, 10, 5)  # optical powers from 10 mW to 1 W

f_rf_r = 3e6
f_rf_a = 1e6
ueq = {N: ion_spacing(N, pi*f_rf_a)[0] for N in range(2, N_max + 1)}

# --- Run power sweep using your tweezed-only function ---
power_results = [[] for _ in N_values]  # List of lists to hold winners for each N
for N in N_values:
    print(f"Working on N = {N}...")
    t_start = time.time()
    for P in P_list:
        winners = run_optimal_mode_selection_tweezed_only_test(
            omega_tweezer,
            linewidths,
            omega_res,
            m,
            mode_calc_r,
            N,
            f_rf_r,
            ueq,
            P,
            w0,
            max_tweezed=1,
        )
        power_results[N - 2].append(winners)
    elapsed = time.time() - t_start
    print(f"  N = {N} done in {elapsed:.1f}s")

Working on N = 2...
  N = 2 done in 0.0s
Working on N = 3...
  N = 3 done in 0.0s
Working on N = 4...
  N = 4 done in 0.0s
Working on N = 5...
  N = 5 done in 0.0s
Working on N = 6...
  N = 6 done in 0.1s
Working on N = 7...
  N = 7 done in 0.9s
Working on N = 8...
  N = 8 done in 5.7s
Working on N = 9...
  N = 9 done in 47.2s
Working on N = 10...
  N = 10 done in 1350.4s
Working on N = 11...


: 

In [1]:
# --- Extract and plot for each N ---
n_panels = N_max - 2  # number of N values = range(2, N_max)
ncols = 4
nrows = (n_panels + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

optimal_powers_W = []
optimal_powers_mW = []
optimal_ratios = []

f_rf_r = 3e6
f_rf_a = 1e6
ueq = {N: ion_spacing(N, pi*f_rf_a)[0] for N in range(2, N_max + 1)}

untweezed_results_local = []
for Ni in range(2, N_max):
    winners = run_optimal_mode_selection_untweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        ueq,
        0,
        w0,
    )
    untweezed_results_local.append((Ni, winners))

untweezed_winners_local = [abs(untweezed_results_local[i - 2][1][0][2]) for i in range(2, N_max)]

for idx, N in enumerate(range(2, N_max)):
    x_tweezed = []
    y_tweezed = []

    for P_val, winners in zip(P_list, power_results[N - 2]):
        if not winners:
            continue
        winner = winners[0]

        if len(winner) >= 3:
            tweezed_ion = winner[0]
            score = winner[2]
        else:
            continue

        if tweezed_ion is None:
            continue

        x_tweezed.append(P_val)
        y_tweezed.append(abs(score))

    x_tweezed = np.array(x_tweezed)
    y_tweezed = np.array(y_tweezed)

    untweezed_ref = untweezed_winners_local[N - 2]

    if len(y_tweezed) > 0:
        ratio = untweezed_ref / y_tweezed
        opt_idx = np.argmin(ratio)

        opt_power_W = x_tweezed[opt_idx]
        opt_power_mW = opt_power_W * 1e3
        opt_ratio = ratio[opt_idx]

        optimal_powers_W.append(opt_power_W)
        optimal_powers_mW.append(opt_power_mW)
        optimal_ratios.append(opt_ratio)

        axes[idx].scatter(x_tweezed * 1e3, ratio, marker='o', color='blue', s=50)
        axes[idx].set_xlabel("Optical power P (mW)", fontsize=10)
        axes[idx].set_ylabel(r"$\tau$ Ratio", fontsize=10)
        axes[idx].set_title(f"N = {N}", fontsize=11, fontweight='bold')
        axes[idx].grid(True, alpha=0.3)
        axes[idx].scatter(
            opt_power_mW, opt_ratio,
            color='red', s=200, marker='*', zorder=5,
            label=f'Opt: {opt_power_mW:.2f} mW',
        )
        axes[idx].legend(fontsize=9)

# Hide unused panels
for ax in axes[n_panels:]:
    ax.set_visible(False)

plt.tight_layout()
plt.suptitle(
    "Power Sweep: Tweezed/Untweezed Ratio for Each N",
    fontsize=14, fontweight='bold', y=1.00,
)
plt.show()

optimal_powers_W = np.array(optimal_powers_W)
optimal_powers_mW = np.array(optimal_powers_mW)
optimal_ratios = np.array(optimal_ratios)

print("\nOptimal Powers Summary:")
print("=" * 60)
for N, P_W, P_mW, ratio in zip(range(2, N_max), optimal_powers_W, optimal_powers_mW, optimal_ratios):
    print(f"N = {N}: P_opt = {P_W:.6e} W = {P_mW:.4f} mW,  Ratio = {ratio:.4f}")
print("=" * 60)

NameError: name 'N_max' is not defined